In [1]:
from sklearn.datasets import load_diabetes

df=load_diabetes()


In [5]:
x,y=df.data,df.target
x,y

(array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]]),
 array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142., 155., 225.,  59., 104., 182.,
   

In [8]:
print(f"Dataset feature name:{str(df.feature_names)}")
print(f"Datset feature size: {str(df.data.shape)}")
print(f"Datset target size: {str(df.target.shape)}")

Dataset feature name:['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Datset feature size: (442, 10)
Datset target size: (442,)


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV,train_test_split,cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(353, 10) (89, 10) (353,) (89,)


In [40]:
lr=LinearRegression()
dt=DecisionTreeRegressor(max_depth=5)
knn=KNeighborsRegressor()

In [41]:
lr.fit(x_train,y_train)
dt.fit(x_train,y_train)
knn.fit(x_train,y_train)

KNeighborsRegressor()

In [42]:
y_pred1=lr.predict(x_test)
y_pred2=dt.predict(x_test)
y_pred3=knn.predict(x_test)

In [43]:
print("R2 Score lr :",r2_score(y_test,y_pred1))
print("R2 Score dt :",r2_score(y_test,y_pred2))
print("R2 Score knn :",r2_score(y_test,y_pred3))

R2 Score lr : 0.4526027629719197
R2 Score dt : 0.2602365893020092
R2 Score knn : 0.43016439526042805


In [93]:
from sklearn.ensemble import BaggingRegressor
from sklearn.svm import SVR

bag_regressor=BaggingRegressor(DecisionTreeRegressor(max_depth=20),random_state=42)
bag_regressor.fit(x_train,y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=20), random_state=42)

In [81]:
y_preds=bag_regressor.predict(x_test)

In [82]:
print("Training Coefficient Of R2:%.3f"%bag_regressor.score(x_train,y_train))
print("Test Coeffiecient of R2:%.3f "%bag_regressor.score(x_test,y_test))

Training Coefficient Of R2:0.906
Test Coeffiecient of R2:0.385 


In [83]:

params={
    'estimator':[None,LinearRegression(),KNeighborsRegressor()],
    'n_estimators':[20,50,100],
    'max_samples':[0.5,1.0],
    'max_features':[0.5,1.0],
    'bootstrap':[True,False],
    'bootstrap_features':[True,False]
}

In [84]:
bag_regressor_grid=GridSearchCV(BaggingRegressor(n_jobs=1),param_grid=params,cv=3,n_jobs=-1,verbose=1)
bag_regressor_grid.fit(x_train,y_train)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


GridSearchCV(cv=3, estimator=BaggingRegressor(n_jobs=1), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'bootstrap_features': [True, False],
                         'estimator': [None, LinearRegression(),
                                       KNeighborsRegressor()],
                         'max_features': [0.5, 1.0], 'max_samples': [0.5, 1.0],
                         'n_estimators': [20, 50, 100]},
             verbose=1)

In [86]:
bag_regressor_grid.best_params_

{'bootstrap': True,
 'bootstrap_features': False,
 'estimator': LinearRegression(),
 'max_features': 1.0,
 'max_samples': 0.5,
 'n_estimators': 100}

In [92]:
print("Train R2 Score:%.3f"%bag_regressor_grid.best_estimator_.score(x_train,y_train))
print("Test R2 Score:%.3f"%bag_regressor_grid.best_estimator_.score(x_test,y_test))
print("Best R2 score through Grid:%.3f"%bag_regressor_grid.best_score_)
print("Best Parameters:",bag_regressor_grid.best_params_)

Train R2 Score:0.527
Test R2 Score:0.453
Best R2 score through Grid:0.491
Best Parameters: {'bootstrap': True, 'bootstrap_features': False, 'estimator': LinearRegression(), 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 100}


In [ ]:
estimators=[('lr',LinearRegression()),('dt',DecisionTreeRegressor(max_depth=20)),('svm',SVR())]
from sklearn.ensemble import VotingRegressor
vc=VotingRegressor(estimators=estimators,verbose=True)
vc.fit(x_train,y_train)

[Voting] ....................... (1 of 3) Processing lr, total=   0.0s
[Voting] ....................... (2 of 3) Processing dt, total=   0.0s
[Voting] ...................... (3 of 3) Processing svm, total=   0.0s


VotingRegressor(estimators=[('lr', LinearRegression()),
                            ('dt', DecisionTreeRegressor(max_depth=20)),
                            ('svm', SVR())],
                verbose=True)

In [97]:
y_pred=vc.predict(x_test)
r2_score(y_test,y_pred)

0.442982444152473



### **Voting Ensemble**
- **What it does**: Combines predictions from **different models** (e.g., decision tree, SVM, logistic regression).
- **How it works**:
  - **Hard Voting**: Majority vote for classification.
  - **Soft Voting**: Averages probabilities for classification.
- **Goal**: Leverages the strengths of diverse models to improve accuracy.
- **Example**: Combining a decision tree, SVM, and logistic regression.

---

### **Bagging**
- **What it does**: Trains **multiple copies of the same model** on different subsets of data (bootstrapped samples).
- **How it works**:
  - Each model is trained on a random subset of data.
  - Predictions are averaged (regression) or majority-voted (classification).
- **Goal**: Reduces overfitting and variance by averaging out errors.
- **Example**: Random Forest (bagging with decision trees).

---

### **Key Difference**
- **Voting**: Uses **different models** and combines their predictions.
- **Bagging**: Uses **the same model** trained on different data subsets.

---

### **When to Use?**
- **Voting**: When you have diverse models with complementary strengths.
- **Bagging**: When your model overfits, and you want to stabilize predictions.

---

This is the **simplest yet deepest explanation**! Let me know if you need more details. 😊